# Lab 00-02: Unity Catalog Basics

**Module:** 00 — Foundations  
**Exam weight:** Foundational — Unity Catalog appears in every exam section  
**UI Sections:** Catalog  
**Estimated Time:** 30–40 minutes  
**Difficulty:** Beginner

## What You Will Build

In this lab, you will learn how Databricks organizes *everything* — your data, your AI models, your functions, even your file uploads — using a system called **Unity Catalog**.

Unity Catalog is the single governance layer across the entire Databricks platform. If you've used databases before, think of it as the "master catalog" that sits above individual databases and controls who can access what.

By the end of this lab, you'll have created your own personal schema (like a personal workspace within the catalog) where all your lab artifacts will live for the rest of this course.

### Why This Matters for the Exam

Unity Catalog is referenced in *every* exam section. Models are registered here. Functions become agent tools here. Tables store your chunks here. Vector Search indexes reference tables here. If you don't understand the three-level namespace (`catalog.schema.object`), you'll struggle with every subsequent topic.

## Mental Model

> **Analogy:** Think of Unity Catalog like a company's filing system. The **catalog** is the filing room (e.g., "Company Data"). A **schema** is a drawer in that room (e.g., "Marketing", "Finance", "GenAI Labs"). Inside each drawer, you have **objects** — individual folders for tables, models, functions, and volumes. To find anything, you need all three levels: filing room → drawer → folder. That's `catalog.schema.object` — the three-level namespace.

Here's the hierarchy visualized:

```
Unity Catalog
└── workspace (catalog)
    ├── default (schema)
    │   ├── my_table (table)
    │   └── my_model (model)
    ├── genai_labs (schema) ← YOUR workspace for this course
    │   ├── document_chunks (table)
    │   ├── rag_chain (model)
    │   └── order_lookup (function)
    └── information_schema (schema, system-generated)
```

## Exam Alert

| Trap | Correct Answer |
|---|---|
| "Unity Catalog only manages tables" | **WRONG** — it manages tables, models, functions, volumes, connections, and more |
| "You need a separate registry for models" | **WRONG** — models are registered directly in Unity Catalog (no separate model registry needed) |
| "Permissions are set at the table level only" | **WRONG** — you can GRANT/REVOKE at catalog, schema, or object level |

> **Exam tip:** If a question mentions "governance" or "access control" for AI assets, the answer almost always involves Unity Catalog.

## Prerequisites

- Completed Lab 00-01 (you have a running cluster)
- Access to a Databricks workspace

## Databricks UI Navigation

Let's start by exploring the Catalog UI:

1. **UI ->** Left nav -> click **Catalog** (the database icon)
2. You'll see a tree view with catalogs at the top level
3. Expand **workspace** — this is the default catalog in most workspaces
4. You'll see schemas inside it (at minimum, `default` and `information_schema`)

Take a moment to click around. Notice that each object shows its type (TABLE, VIEW, MODEL, FUNCTION, VOLUME) and owner.

In [ ]:
# ------------------------------------------------------------------
# Setup: No special installs needed — Unity Catalog is built in
# ------------------------------------------------------------------

# Verify cluster connection and check current catalog
print(f"Spark version: {spark.version}")
print(f"Current catalog: {spark.catalog.currentCatalog()}")
print(f"Current database: {spark.catalog.currentDatabase()}")

**Expected output:**
```
Spark version: 15.x.x
Current catalog: workspace
Current database: default
```

**What just happened:** You confirmed your cluster is connected and that you're working in the `workspace` catalog by default. The "current database" is Spark's term for the current schema.

## Step 1: Explore the Three-Level Namespace

Every object in Databricks has a **fully qualified name** with three parts:

```
catalog.schema.object
```

This is like a mailing address: **Country.City.Street**. You need all three to unambiguously find anything.

- **Catalog** = the top-level container (your workspace has `workspace`)
- **Schema** (also called database) = a logical grouping inside a catalog
- **Object** = a table, view, model, function, or volume

Let's explore what catalogs and schemas exist in your workspace.

In [ ]:
# ------------------------------------------------------------------
# Step 1: Explore catalogs and schemas
# ------------------------------------------------------------------

# List all catalogs you have access to
print("=== Available Catalogs ===")
catalogs = spark.sql("SHOW CATALOGS")
catalogs.show(truncate=False)

# List schemas in the 'workspace' catalog
print("\n=== Schemas in 'workspace' catalog ===")
schemas = spark.sql("SHOW SCHEMAS IN workspace")
schemas.show(truncate=False)

**Expected output:**
```
=== Available Catalogs ===
+--------+
| catalog|
+--------+
| workspace|
| system |
+--------+

=== Schemas in 'workspace' catalog ===
+--------------------+
| databaseName       |
+--------------------+
| default            |
| information_schema |
+--------------------+
```

**What just happened:** You discovered that your workspace has catalogs (`workspace` for your data, `system` for metadata). Inside `workspace`, there's a `default` schema and the system-generated `information_schema`. You'll add your own schema next.

> **Note:** In the Databricks UI, you can see this same information by clicking **Catalog** in the left nav and expanding the tree. Try it now — compare what you see in the UI with what the SQL returned.

## Step 2: Create Your Personal Schema

For the rest of this course, all your lab artifacts (tables, models, functions) will live in a schema called `genai_labs` inside the `workspace` catalog.

Think of this as creating your own drawer in the filing cabinet. Everything you build goes in this drawer, keeping it organized and separate from other users.

**UI ->** After running the code below, go to Left nav -> **Catalog** -> expand **workspace** -> you should see `genai_labs` appear in the tree.

In [ ]:
# ------------------------------------------------------------------
# Step 2: Create your personal schema for all lab work
# ------------------------------------------------------------------

# Create the schema (IF NOT EXISTS prevents errors if you run this twice)
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.genai_labs
    COMMENT 'Personal schema for GenAI Engineer Associate lab work'
""")

# Set it as your default so you don't have to type 'workspace.genai_labs' every time
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA genai_labs")

# Verify
print(f"Current catalog: {spark.catalog.currentCatalog()}")
print(f"Current schema:  {spark.catalog.currentDatabase()}")
print("\nSchema 'workspace.genai_labs' is ready! All lab artifacts will go here.")

**Expected output:**
```
Current catalog: workspace
Current schema:  genai_labs

Schema 'workspace.genai_labs' is ready! All lab artifacts will go here.
```

**What just happened:** You created a schema called `genai_labs` inside the `workspace` catalog. The `USE CATALOG` and `USE SCHEMA` commands set your defaults — now any unqualified table name like `my_table` automatically resolves to `workspace.genai_labs.my_table`.

> **Tip:** You only need to run `USE CATALOG` and `USE SCHEMA` once per notebook session. Many Databricks users add this to the first cell of every notebook.

## Step 3: Understand What Lives in Unity Catalog

Here's the key insight that separates Databricks from other platforms: **Unity Catalog manages far more than just tables.** It's the single governance layer for ALL assets:

| Asset Type | What It Is | Example from This Course |
|---|---|---|
| **TABLE** | Structured data (Delta format) | `document_chunks` — your RAG knowledge base |
| **MODEL** | Registered ML/AI models | `rag_chain` — your deployed RAG application |
| **FUNCTION** | Python/SQL functions | `order_lookup` — an agent tool |
| **VOLUME** | Unstructured files (PDFs, images) | `source_docs` — raw documents before processing |
| **CONNECTION** | External data source credentials | Connections to S3, ADLS, external APIs |

This is why the exam says "Unity Catalog is the governance layer for both data and AI assets." When you register a model with MLflow, it goes into Unity Catalog. When you create a function for an agent tool, it goes into Unity Catalog. When you upload documents for RAG, they go into a Volume in Unity Catalog.

In [ ]:
# ------------------------------------------------------------------
# Step 3: Create example assets to see the different object types
# ------------------------------------------------------------------

# Create a sample table
spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.genai_labs.sample_assets (
        id INT,
        name STRING,
        description STRING
    )
    COMMENT 'Sample table to demonstrate Unity Catalog asset types'
""")

# Insert some sample data
spark.sql("""
    INSERT OVERWRITE workspace.genai_labs.sample_assets VALUES
    (1, 'Delta Table', 'Stores structured data like document chunks'),
    (2, 'ML Model', 'Registered AI models like your RAG chain'),
    (3, 'UC Function', 'Python/SQL functions that become agent tools'),
    (4, 'Volume', 'Unstructured file storage for PDFs, images, etc.')
""")

# Create a Volume (for storing unstructured files like PDFs)
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.genai_labs.source_docs
    COMMENT 'Volume for storing source documents (PDFs, images, HTML)'
""")

# List everything in our schema
print("=== Tables in workspace.genai_labs ===")
spark.sql("SHOW TABLES IN workspace.genai_labs").show(truncate=False)

print("\n=== Volumes in workspace.genai_labs ===")
spark.sql("SHOW VOLUMES IN workspace.genai_labs").show(truncate=False)

**Expected output:**
```
=== Tables in workspace.genai_labs ===
+----------+--------------+-----------+
| database | tableName    | isTemporary|
+----------+--------------+-----------+
|genai_labs| sample_assets| false     |
+----------+--------------+-----------+

=== Volumes in workspace.genai_labs ===
+--------+-----------+-----------+-----------+
|catalog |schema     |volume_name|volume_type|
+--------+-----------+-----------+-----------+
|workspace|genai_labs |source_docs|MANAGED    |
+--------+-----------+-----------+-----------+
```

**What just happened:** You created two different types of Unity Catalog assets — a table and a volume — both living in your `genai_labs` schema. Go to **Catalog** in the left nav and expand `workspace.genai_labs` to see both listed.

Notice the Volume type is `MANAGED` — Databricks manages the underlying storage. Later in Module 02, you'll upload source documents to this volume.

## Step 4: Permissions — GRANT and REVOKE

Unity Catalog uses a SQL-based permission model. You control who can access what using `GRANT` and `REVOKE` statements.

The permission hierarchy follows the namespace:
- Grant on a **catalog** → applies to all schemas and objects inside it
- Grant on a **schema** → applies to all objects in that schema
- Grant on an **object** → applies to just that table/model/function

This is like giving someone a key:
- Key to the **building** (catalog) = access to everything
- Key to a **floor** (schema) = access to everything on that floor
- Key to a **room** (object) = access to just that room

Common permissions:
| Permission | What It Allows |
|---|---|
| `USE CATALOG` | See the catalog exists and list its schemas |
| `USE SCHEMA` | See the schema exists and list its objects |
| `SELECT` | Read data from a table |
| `MODIFY` | Insert, update, delete data in a table |
| `CREATE TABLE` | Create new tables in a schema |
| `ALL PRIVILEGES` | Full access (use sparingly!) |

> **Note:** On Databricks Community Edition or personal workspaces, you may not have permission to run GRANT/REVOKE. That's OK — read through the syntax to understand it for the exam.

In [ ]:
# ------------------------------------------------------------------
# Step 4: Practice GRANT and REVOKE syntax
# ------------------------------------------------------------------

# NOTE: These commands may fail on Community Edition / limited workspaces.
# That's OK — the important thing is to understand the syntax for the exam.

# Example: Grant SELECT on a table to a user
# GRANT SELECT ON TABLE workspace.genai_labs.sample_assets TO `user@example.com`;

# Example: Grant USE SCHEMA to a group
# GRANT USE SCHEMA ON SCHEMA workspace.genai_labs TO `data_scientists`;

# Example: Revoke access
# REVOKE SELECT ON TABLE workspace.genai_labs.sample_assets FROM `user@example.com`;

# Let's check the current grants on our schema (this usually works)
try:
    grants = spark.sql("SHOW GRANTS ON SCHEMA workspace.genai_labs")
    grants.show(truncate=False)
except Exception as e:
    print(f"Note: SHOW GRANTS requires admin privileges.")
    print(f"Error: {e}")
    print("\nDon't worry — learn the syntax, that's what the exam tests:")
    print("  GRANT <permission> ON <object_type> <object_name> TO `<principal>`")
    print("  REVOKE <permission> ON <object_type> <object_name> FROM `<principal>`")

**Expected output** (varies by workspace permissions):
```
Either a grants table showing your permissions, or:

Note: SHOW GRANTS requires admin privileges.
Don't worry — learn the syntax, that's what the exam tests:
  GRANT <permission> ON <object_type> <object_name> TO `<principal>`
  REVOKE <permission> ON <object_type> <object_name> FROM `<principal>`
```

**What just happened:** You explored the GRANT/REVOKE syntax. The exam tests whether you know this pattern — the actual ability to run these commands depends on your workspace role. The key pattern to memorize:

```sql
GRANT <permission> ON <object_type> <fully_qualified_name> TO `<principal>`
```

## Step 5: Describe and Inspect Objects

A powerful feature of Unity Catalog is its rich metadata. Every object has a description, owner, creation time, and more. This metadata is what makes governance possible — you can always answer "who created this?" and "what is this for?"

The `DESCRIBE` command is your best friend for inspecting objects.

In [ ]:
# ------------------------------------------------------------------
# Step 5: Inspect object metadata
# ------------------------------------------------------------------

# Describe the schema itself
print("=== Schema Description ===")
spark.sql("DESCRIBE SCHEMA workspace.genai_labs").show(truncate=False)

# Describe a table — shows columns, types, and comments
print("\n=== Table Description ===")
spark.sql("DESCRIBE TABLE workspace.genai_labs.sample_assets").show(truncate=False)

# Extended describe — shows storage location, owner, created time, table properties
print("\n=== Table Extended Details ===")
spark.sql("DESCRIBE TABLE EXTENDED workspace.genai_labs.sample_assets").show(truncate=False)

**Expected output:** Three tables showing:
1. Schema metadata (name, comment, location, owner)
2. Table columns (id INT, name STRING, description STRING)
3. Extended details (storage location, provider=delta, table properties, owner, created timestamp)

**What just happened:** You used `DESCRIBE` to inspect metadata at both the schema and table level. Notice how the extended describe shows `Provider: delta` — confirming this is a Delta table. It also shows the `Owner`, which is the user who created it.

## Step 6: Clean Up (Keep the Schema!)

We'll remove the sample table but **keep the schema and volume** — they'll be used in every subsequent lab.

In [ ]:
# ------------------------------------------------------------------
# Step 6: Clean up sample table (keep schema and volume)
# ------------------------------------------------------------------

spark.sql("DROP TABLE IF EXISTS workspace.genai_labs.sample_assets")

print("Cleaned up sample_assets table.")
print("Kept: workspace.genai_labs schema and source_docs volume")
print("\nThese will be used in all subsequent labs.")

**Expected output:**
```
Cleaned up sample_assets table.
Kept: workspace.genai_labs schema and source_docs volume

These will be used in all subsequent labs.
```

## Exam Tips & Traps

**Quick-scan reference — review these before the exam:**

| # | Topic | Trap | Correct Answer |
|---|---|---|---|
| 1 | Namespace | "Tables use a two-level namespace" | **WRONG** — always three levels: `catalog.schema.table` |
| 2 | Asset types | "Models need a separate model registry" | **WRONG** — models are registered directly in Unity Catalog via MLflow |
| 3 | Permissions | "GRANT on a table applies to the schema" | **WRONG** — permissions inherit downward: catalog → schema → object, not upward |
| 4 | Volumes | "Volumes store structured data" | **WRONG** — Volumes store *unstructured* files (PDFs, images). Tables store structured data |
| 5 | Functions | "Agent tools are separate from Unity Catalog" | **WRONG** — UC functions automatically become agent tools via managed MCP (Lab 04-07) |

> **Pattern to watch for:** When the exam says "governance," "access control," or "centrally managed," the answer almost always involves Unity Catalog.

## Key Takeaways

### Core Concepts
- Unity Catalog uses a **three-level namespace**: `catalog.schema.object`
- It governs ALL asset types: tables, models, functions, volumes, connections
- Permissions use SQL syntax: `GRANT <perm> ON <type> <name> TO <principal>`
- Permissions inherit downward: catalog → schema → object

### Databricks-Specific
- **UI ->** Left nav -> **Catalog** to browse and manage all assets
- `USE CATALOG` and `USE SCHEMA` set your defaults for the session
- `DESCRIBE EXTENDED` reveals rich metadata: owner, created time, properties
- Volumes are for unstructured files; Tables are for structured data

### Exam Essentials
- Unity Catalog = the single governance layer for data AND AI assets
- The three-level namespace (`catalog.schema.object`) appears in many questions
- Models, functions, and tables ALL live in Unity Catalog — not separate registries
- `GRANT` at schema level applies to all objects in that schema

---

## Next Lab

**Lab 00-03: Your First LLM Call** — you'll use the Playground to chat with a foundation model, then call it programmatically from a notebook.

> **Keep these artifacts:** The `workspace.genai_labs` schema and `source_docs` volume you created here are used in every subsequent lab. Do not delete them.